# Fine-tuned design self-consistency (R1-h)

For each fine-tuned design we fold the designed sequence (already done, ColabFold AF2,
single-sequence) and measure how well it reproduces the **wild-type backbone** it was
designed for:

- **scTM** — TM-score of the design's structure vs the wild-type structure (1 = identical fold; >0.5 = same fold)
- **scRMSD** — Cα RMSD after optimal superposition (Å; lower = better)
- **pLDDT** — the design's own AlphaFold confidence (from the PDB B-factors)

Both the design and the wild type are folded by the **same predictor** (ColabFold,
single-sequence), so the comparison is predictor-consistent. The headline is the
**paired base-vs-fine-tuned** difference (same wild-type reference for both), which shows
whether the surface-charge fine-tuning costs any structural compatibility.

**Upload `ft_structures.zip`** (625 rank-1 PDBs: 600 designs + 25 wild types) when prompted.

In [ ]:
# @title 1. Install + imports
!pip -q install tmtools biotite
import os, glob, re, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from tmtools import tm_align
from tmtools.io import get_structure, get_residue_data
from scipy.stats import wilcoxon

In [ ]:
# @title 2. Upload ft_structures.zip and unzip
from google.colab import files
files.upload()                                  # pick ft_structures.zip
!unzip -o -q ft_structures.zip -d /content/structures
pdbs = glob.glob("/content/structures/**/*rank_001*.pdb", recursive=True)
print(len(pdbs), "structures found (expect 625)")

In [ ]:
# @title 3. Parsers — Cα coords + sequence + mean pLDDT
def parse_ca(path):
    chain = next(get_structure(path).get_chains())
    coords, seq = get_residue_data(chain)       # (N,3) Cα coords, 1-letter sequence
    return np.asarray(coords, float), seq

def mean_plddt(path):                            # ColabFold stores pLDDT in the B-factor column
    vals = [float(l[60:66]) for l in open(path)
            if l.startswith("ATOM") and l[12:16].strip() == "CA"]
    return float(np.mean(vals)) if vals else np.nan

def fold_id(path):                               # <uniprot>__<model>__s<i>  or  <uniprot>__WT
    return os.path.basename(path).split("_unrelaxed_rank")[0]

In [ ]:
# @title 4. Self-consistency: each design vs its wild-type backbone
struct = {fold_id(p): p for p in pdbs}
wt = {k[:-4]: v for k, v in struct.items() if k.endswith("__WT")}   # uniprot -> WT pdb
rows = []
for fid, p in struct.items():
    if fid.endswith("__WT"):
        continue
    m = re.match(r"(.+?)__(.+?)__s(\d+)$", fid)
    if not m or m.group(1) not in wt:
        continue
    uni, model, s = m.group(1), m.group(2), int(m.group(3))
    dc, ds = parse_ca(p)
    wc, ws = parse_ca(wt[uni])
    r = tm_align(dc, wc, ds, ws)                  # design (chain1) vs wild type (chain2)
    rows.append(dict(uniprot_id=uni, model=model, sample_idx=s,
                     scTM=r.tm_norm_chain2, scRMSD=r.rmsd, pLDDT=mean_plddt(p)))
sc = pd.DataFrame(rows)
print(sc.shape[0], "designs scored")
sc.head()

In [ ]:
# @title 5. Base vs fine-tuned — summary, paired significance, figure
order = ["ProteinMPNN_v002", "AlkSecMPNN", "AcidSecMPNN"]
sc["model"] = pd.Categorical(sc["model"], order, ordered=True)

print("=== per-model self-consistency ===")
print(sc.groupby("model")[["scTM", "scRMSD", "pLDDT"]].agg(["mean", "median"]).round(3))

print("\n=== fine-tuned vs base (paired by template, Wilcoxon signed-rank) ===")
base = sc[sc.model == "ProteinMPNN_v002"].groupby("uniprot_id")[["scTM","scRMSD","pLDDT"]].mean()
for ft in ["AlkSecMPNN", "AcidSecMPNN"]:
    f = sc[sc.model == ft].groupby("uniprot_id")[["scTM","scRMSD","pLDDT"]].mean()
    c = base.index.intersection(f.index)
    for met in ["scTM", "scRMSD", "pLDDT"]:
        d = (f.loc[c, met] - base.loc[c, met])
        try:    p = wilcoxon(f.loc[c, met], base.loc[c, met]).pvalue
        except ValueError: p = float("nan")
        print(f"  {ft:15s} {met:6s}  delta={d.mean():+.3f}  p={p:.3f}")

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for a, met, lab in zip(ax, ["scTM","scRMSD","pLDDT"],
                       ["self-consistency TM-score","self-consistency RMSD (A)","design pLDDT"]):
    sc.boxplot(column=met, by="model", ax=a, grid=False)
    a.set_title(lab); a.set_xlabel("")
    a.set_xticklabels([t.get_text().replace("ProteinMPNN_v002","base")
                       for t in a.get_xticklabels()], rotation=15)
plt.suptitle("Fine-tuned design self-consistency (design vs wild-type backbone; 25 templates x 8 designs)")
plt.tight_layout()
plt.savefig("ft_self_consistency.png", dpi=150, bbox_inches="tight")
sc.to_csv("ft_self_consistency.csv", index=False)
print("\nSaved ft_self_consistency.csv + ft_self_consistency.png")